In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import optuna
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

INPUT_FILE = '/kaggle/working/DeepSOC_predictors_table_final.csv'
OUTPUT_BASE_DIR = '/kaggle/working/ML_SOC_Results'
RESPONSE_VARIABLE = 'DeepSOC30_100'
LUV2_VARIABLE = 'Ecosystem'
SOC_UPPER_LIMIT = 1000
N_OUTER_FOLDS = 10
N_OUTER_REPEATS = 10
N_INNER_FOLDS = 4
OPTUNA_TRIALS = 100

NUMERIC_PREDICTORS = [
    'NDVI',
    'NDWI',
    'NIRI',
    'elevation',
    'slope',
    'mat',
    'map',
    'treecover',
]

CATEGORICAL_PREDICTORS = [
    'lulc',
    'soil_type',
]

MODEL_REGISTRY = {
    'RandomForest': RandomForestRegressor,
    'XGBoost': xgb.XGBRegressor,
}

FIXED_PARAMS = {
    'RandomForest': dict(random_state=42, n_jobs=-1),
    'XGBoost': dict(random_state=42, n_jobs=-1, verbosity=0),
}


def load_data(file_path, response_variable, numeric_preds, categorical_preds,
              luv2_variable=None, soc_upper_limit=None):

    print("=" * 70)
    print(f"Response variable : {response_variable}")
    print("=" * 70)

    df = pd.read_csv(file_path, sep=',')
    print(f"Raw shape         : {df.shape}")
    print(f"Columns           : {list(df.columns)}")

    df.replace(-9999, np.nan, inplace=True)

    before = len(df)
    df = df.dropna(subset=[response_variable])
    print(f"Dropped {before - len(df)} rows where response is NaN")

    if soc_upper_limit is not None:
        before = len(df)
        df = df[df[response_variable] < soc_upper_limit].copy()
        print(f"Filtered {before - len(df)} rows with {response_variable} >= {soc_upper_limit}")

    print(f"Working dataset   : {len(df)} rows")

    if luv2_variable and luv2_variable in df.columns:
        print(f"LUv2 classes found: {sorted(df[luv2_variable].dropna().unique())}")
    else:
        print(f"[WARN] LUv2 column '{luv2_variable}' not found in CSV")

    available_num = [c for c in numeric_preds if c in df.columns]
    available_cat = [c for c in categorical_preds if c in df.columns]
    missing_num = [c for c in numeric_preds if c not in df.columns]
    missing_cat = [c for c in categorical_preds if c not in df.columns]

    if missing_num:
        print(f"[WARN] Numeric predictors not found    : {missing_num}")
    if missing_cat:
        print(f"[WARN] Categorical predictors not found: {missing_cat}")

    extra_cols = [luv2_variable] if luv2_variable and luv2_variable in df.columns else []
    keep_cols = list(dict.fromkeys([response_variable] + available_num + available_cat + extra_cols))
    df = df[keep_cols].copy()

    for col in available_num:
        if df[col].isnull().any():
            n_miss = df[col].isnull().sum()
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)
            print(f"  Imputed {n_miss:>4} NaN in '{col}' with median={median_val:.4f}")

    encoders = {}
    encoded_cols = []
    for col in available_cat:
        df[col] = df[col].astype(str).fillna('Unknown')
        le = LabelEncoder()
        enc = f'{col}_enc'
        df[enc] = le.fit_transform(df[col])
        encoders[col] = le
        encoded_cols.append(enc)
        print(f"Encoded '{col}': {len(le.classes_)} classes → {list(le.classes_)}")

    feature_cols = available_num + encoded_cols

    print(f"\nFinal dataset shape : {df.shape}")
    print(f"Features used       : {len(feature_cols)}")
    print(f"\n{response_variable} statistics:")
    print(df[response_variable].describe().round(4))

    return df, feature_cols, encoders


def calc_metrics(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    if len(y_true) < 2:
        return dict(n=len(y_true), rmse=np.nan, r2=np.nan, mae=np.nan,
                    bias=np.nan, rpd=np.nan, rpiq=np.nan, ccc=np.nan)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    sd = np.std(y_true)
    iqr = np.percentile(y_true, 75) - np.percentile(y_true, 25)
    bias = float(np.mean(y_pred - y_true))
    rpd = sd / rmse if rmse > 0 else 0.0
    rpiq = iqr / rmse if rmse > 0 else 0.0
    ccc_num = 2 * np.cov(y_true, y_pred, ddof=0)[0, 1]
    ccc_den = (np.var(y_true, ddof=0) + np.var(y_pred, ddof=0) +
               (np.mean(y_true) - np.mean(y_pred)) ** 2)
    ccc = ccc_num / ccc_den if ccc_den > 0 else 0.0
    return dict(n=int(len(y_true)), rmse=rmse, r2=r2, mae=mae,
                bias=bias, rpd=rpd, rpiq=rpiq, ccc=ccc)


def make_objective(model_name, X_np, y_arr, n_inner_folds):
    def objective(trial):
        if model_name == 'RandomForest':
            params = dict(
                n_estimators=trial.suggest_int('n_estimators', 100, 1000),
                max_depth=trial.suggest_int('max_depth', 5, 30),
                min_samples_split=trial.suggest_int('min_samples_split', 2, 15),
                min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 8),
                max_features=trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
                random_state=42, n_jobs=-1,
            )
            model_cls = RandomForestRegressor
        elif model_name == 'XGBoost':
            params = dict(
                n_estimators=trial.suggest_int('n_estimators', 100, 1000),
                max_depth=trial.suggest_int('max_depth', 3, 10),
                learning_rate=trial.suggest_float('learning_rate', 0.01, 0.30, log=True),
                subsample=trial.suggest_float('subsample', 0.6, 1.0),
                colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
                reg_alpha=trial.suggest_float('reg_alpha', 0.0, 2.0),
                reg_lambda=trial.suggest_float('reg_lambda', 0.5, 5.0),
                random_state=42, n_jobs=-1, verbosity=0,
            )
            model_cls = xgb.XGBRegressor
        else:
            raise ValueError(model_name)

        inner_kf = KFold(n_splits=n_inner_folds, shuffle=True, random_state=0)
        scores = []
        for i, (tr, va) in enumerate(inner_kf.split(X_np)):
            scaler = StandardScaler()
            Xtr = scaler.fit_transform(X_np[tr])
            Xva = scaler.transform(X_np[va])
            model = model_cls(**params)
            model.fit(Xtr, y_arr[tr])
            fold_rmse = np.sqrt(mean_squared_error(y_arr[va], model.predict(Xva)))
            scores.append(fold_rmse)

            trial.report(float(np.mean(scores)), i)
            if trial.should_prune():
                raise optuna.TrialPruned()

        return float(np.mean(scores))
    return objective


def inner_hyperparam_search(model_name, X_train, y_train, n_inner_folds, n_trials):
    sampler = optuna.samplers.TPESampler(seed=42)
    pruner = optuna.pruners.MedianPruner(n_warmup_steps=1)
    study = optuna.create_study(direction='minimize', sampler=sampler, pruner=pruner)
    study.optimize(
        make_objective(model_name, X_train, y_train, n_inner_folds),
        n_trials=n_trials,
        show_progress_bar=False,
    )
    return {**FIXED_PARAMS[model_name], **study.best_params}, study.best_value


def nested_repeated_kfold_cv(model_name, model_class, X_df, y_series, luv2_series=None,
                              n_outer_folds=10, n_outer_repeats=10, n_inner_folds=4,
                              n_trials=100):

    X_np_full = X_df.values
    y_np_full = y_series.values

    fold_metrics = []
    all_obs, all_pred, all_repeat, all_fold, all_luv2 = [], [], [], [], []
    best_params_per_fold = []
    total_folds = n_outer_folds * n_outer_repeats
    run = 0

    for rep in range(n_outer_repeats):
        outer_kf = KFold(n_splits=n_outer_folds, shuffle=True, random_state=rep * 100 + 42)
        for fold, (tr_idx, va_idx) in enumerate(outer_kf.split(X_np_full), 1):
            run += 1
            print(f"    Outer fold {run}/{total_folds}  (rep {rep+1}, fold {fold})")

            X_tr_outer = X_np_full[tr_idx]
            y_tr_outer = y_np_full[tr_idx]
            X_va_outer = X_np_full[va_idx]
            y_va_outer = y_np_full[va_idx]

            best_params, best_inner_rmse = inner_hyperparam_search(
                model_name, X_tr_outer, y_tr_outer, n_inner_folds, n_trials
            )
            best_params_per_fold.append({
                'repeat': rep + 1, 'fold': fold,
                'best_inner_rmse': best_inner_rmse, **best_params,
            })

            scaler = StandardScaler()
            X_tr_scaled = scaler.fit_transform(X_tr_outer)
            X_va_scaled = scaler.transform(X_va_outer)

            model = model_class(**best_params)
            model.fit(X_tr_scaled, y_tr_outer)
            preds = model.predict(X_va_scaled)

            m = calc_metrics(y_va_outer, preds)
            m['repeat'] = rep + 1
            m['fold'] = fold
            fold_metrics.append(m)

            all_obs.extend(y_va_outer.tolist())
            all_pred.extend(preds.tolist())
            all_repeat.extend([rep + 1] * len(preds))
            all_fold.extend([fold] * len(preds))
            all_luv2.extend(
                luv2_series.iloc[va_idx].tolist() if luv2_series is not None else ['Unknown'] * len(preds)
            )

    metric_keys = ['rmse', 'r2', 'mae', 'bias', 'rpd', 'rpiq', 'ccc']
    mean_metrics = {k: float(np.mean([f[k] for f in fold_metrics])) for k in metric_keys}
    std_metrics = {f'{k}_std': float(np.std([f[k] for f in fold_metrics])) for k in metric_keys}

    repeat_summaries = []
    for rep in range(1, n_outer_repeats + 1):
        rep_folds = [f for f in fold_metrics if f['repeat'] == rep]
        rs = {'repeat': rep}
        for k in metric_keys:
            rs[f'mean_{k}'] = float(np.mean([f[k] for f in rep_folds]))
            rs[f'std_{k}'] = float(np.std([f[k] for f in rep_folds]))
        repeat_summaries.append(rs)

    overall = calc_metrics(all_obs, all_pred)

    predictions_df = pd.DataFrame({
        'observed': all_obs, 'predicted': all_pred,
        'repeat': all_repeat, 'fold': all_fold, 'LUv2': all_luv2,
    })

    return {
        'fold_metrics': fold_metrics,
        'mean': mean_metrics,
        'std': std_metrics,
        'repeat_summaries': repeat_summaries,
        'overall': overall,
        'predictions_df': predictions_df,
        'best_params_per_fold': pd.DataFrame(best_params_per_fold),
    }


def compute_luv2_metrics(predictions_df):
    rows = []
    for lu in sorted(predictions_df['LUv2'].dropna().unique()):
        sub = predictions_df[predictions_df['LUv2'] == lu]
        if len(sub) < 2:
            continue
        m = calc_metrics(sub['observed'].values, sub['predicted'].values)
        rows.append({'LUv2': lu, 'n': m['n'],
                     'R2': round(m['r2'], 4), 'RMSE': round(m['rmse'], 4),
                     'MAE': round(m['mae'], 4), 'Bias': round(m['bias'], 4),
                     'RPD': round(m['rpd'], 4), 'RPIQ': round(m['rpiq'], 4),
                     'CCC': round(m['ccc'], 4)})
    m_all = calc_metrics(predictions_df['observed'].values, predictions_df['predicted'].values)
    rows.append({'LUv2': 'ALL', 'n': m_all['n'],
                 'R2': round(m_all['r2'], 4), 'RMSE': round(m_all['rmse'], 4),
                 'MAE': round(m_all['mae'], 4), 'Bias': round(m_all['bias'], 4),
                 'RPD': round(m_all['rpd'], 4), 'RPIQ': round(m_all['rpiq'], 4),
                 'CCC': round(m_all['ccc'], 4)})
    return pd.DataFrame(rows)


def print_luv2_table(lu_df, model_name):
    print(f"\n  ── Per-LUv2 metrics ({model_name}) ──")
    hdr = (f"  {'LUv2':<30} {'n':>5} {'R2':>7} {'RMSE':>8} "
           f"{'MAE':>8} {'Bias':>8} {'RPD':>7} {'RPIQ':>7} {'CCC':>7}")
    sep = '  ' + '─' * (len(hdr) - 2)
    print(sep); print(hdr); print(sep)
    for _, row in lu_df.iterrows():
        print(f"  {str(row['LUv2']):<30} {int(row['n']):>5} "
              f"{row['R2']:>7.4f} {row['RMSE']:>8.4f} {row['MAE']:>8.4f} "
              f"{row['Bias']:>8.4f} {row['RPD']:>7.4f} {row['RPIQ']:>7.4f} {row['CCC']:>7.4f}")
    print(sep)


def run_pipeline(file_path, response_variable, numeric_preds, categorical_preds,
                  output_base_dir, luv2_variable=None, soc_upper_limit=None,
                  n_outer_folds=10, n_outer_repeats=10, n_inner_folds=4, n_trials=100):

    df, feature_cols, encoders = load_data(
        file_path, response_variable, numeric_preds, categorical_preds,
        luv2_variable=luv2_variable, soc_upper_limit=soc_upper_limit
    )

    X = df[feature_cols].copy()
    y = df[response_variable].copy()
    luv2_series = df[luv2_variable].copy() if luv2_variable and luv2_variable in df.columns else None

    out_dir = os.path.join(output_base_dir, f'results_{response_variable}')
    os.makedirs(out_dir, exist_ok=True)

    all_results = {}
    summary_rows = []
    lu_rows_all = []

    for model_name, model_class in MODEL_REGISTRY.items():
        print(f"\n{'─'*65}\n  {model_name}\n{'─'*65}")
        print(f"  Nested CV: outer {n_outer_folds}-fold x {n_outer_repeats}-repeat, "
              f"inner {n_inner_folds}-fold, {n_trials} Optuna trials with pruning")

        cv = nested_repeated_kfold_cv(
            model_name, model_class, X, y, luv2_series=luv2_series,
            n_outer_folds=n_outer_folds, n_outer_repeats=n_outer_repeats,
            n_inner_folds=n_inner_folds, n_trials=n_trials,
        )

        print(f"\n  ── Aggregated CV metrics (mean ± std across {n_outer_folds*n_outer_repeats} outer folds) ──")
        for k in ['r2', 'rmse', 'mae', 'bias', 'rpiq', 'ccc']:
            print(f"  Mean {k.upper():<6}: {cv['mean'][k]:.4f}  ± {cv['std'][f'{k}_std']:.4f}")

        print(f"\n  ── Per-repeat summaries ──")
        hdr = (f"  {'Rep':>4} {'mean_R2':>9} {'std_R2':>8} "
               f"{'mean_RMSE':>10} {'std_RMSE':>9} {'mean_CCC':>9}")
        print(hdr)
        print('  ' + '─' * (len(hdr) - 2))
        for rs in cv['repeat_summaries']:
            print(f"  {rs['repeat']:>4}  {rs['mean_r2']:>8.4f}  {rs['std_r2']:>8.4f}"
                  f"  {rs['mean_rmse']:>9.4f}  {rs['std_rmse']:>8.4f}"
                  f"  {rs['mean_ccc']:>8.4f}")

        print(f"\n  ── Pooled-overall metrics ──")
        for k in ['r2', 'rmse', 'mae', 'bias', 'rpd', 'rpiq', 'ccc']:
            print(f"  Overall {k.upper():<6}: {cv['overall'][k]:.4f}")

        lu_df = compute_luv2_metrics(cv['predictions_df'])
        lu_df.insert(0, 'Model', model_name)
        print_luv2_table(lu_df, model_name)
        lu_rows_all.append(lu_df)

        print(f"\n  Refitting final model on full dataset with Optuna "
              f"({n_trials} trials, {n_inner_folds}-fold inner CV, pruning)…")
        final_best_params, final_best_rmse = inner_hyperparam_search(
            model_name, X.values, y.values, n_inner_folds, n_trials
        )
        print(f"  Final best inner RMSE : {final_best_rmse:.4f}")
        print(f"  Final best params     : {final_best_params}")

        scaler_final = StandardScaler()
        X_scaled = scaler_final.fit_transform(X.values)
        final_model = model_class(**final_best_params)
        final_model.fit(X_scaled, y)

        feat_imp = None
        if hasattr(final_model, 'feature_importances_'):
            feat_imp = sorted(zip(feature_cols, final_model.feature_importances_),
                              key=lambda x: x[1], reverse=True)
            print(f"\n  Top 10 features:")
            for feat, imp in feat_imp[:10]:
                print(f"    {feat:<35}: {imp:.4f}")

        all_results[model_name] = {
            'model': final_model,
            'scaler': scaler_final,
            'best_params': final_best_params,
            'cv': cv,
            'feat_imp': feat_imp,
            'lu_metrics_df': lu_df,
        }

        summary_rows.append({
            'Model': model_name,
            'Mean_CV_R2': round(cv['mean']['r2'], 4),
            'Std_CV_R2': round(cv['std']['r2_std'], 4),
            'Mean_CV_RMSE': round(cv['mean']['rmse'], 4),
            'Std_CV_RMSE': round(cv['std']['rmse_std'], 4),
            'Mean_CV_MAE': round(cv['mean']['mae'], 4),
            'Std_CV_MAE': round(cv['std']['mae_std'], 4),
            'Mean_CV_Bias': round(cv['mean']['bias'], 4),
            'Std_CV_Bias': round(cv['std']['bias_std'], 4),
            'Mean_CV_RPIQ': round(cv['mean']['rpiq'], 4),
            'Std_CV_RPIQ': round(cv['std']['rpiq_std'], 4),
            'Mean_CV_CCC': round(cv['mean']['ccc'], 4),
            'Std_CV_CCC': round(cv['std']['ccc_std'], 4),
            'Overall_R2': round(cv['overall']['r2'], 4),
            'Overall_RMSE': round(cv['overall']['rmse'], 4),
            'Overall_MAE': round(cv['overall']['mae'], 4),
            'Overall_Bias': round(cv['overall']['bias'], 4),
            'Overall_RPD': round(cv['overall']['rpd'], 4),
            'Overall_RPIQ': round(cv['overall']['rpiq'], 4),
            'Overall_CCC': round(cv['overall']['ccc'], 4),
        })

    summary_df = (pd.DataFrame(summary_rows)
                    .sort_values(['Overall_R2', 'Overall_RMSE'], ascending=[False, True])
                    .reset_index(drop=True))
    best_name = summary_df.iloc[0]['Model']
    lu_all_df = pd.concat(lu_rows_all, ignore_index=True)

    print("\n" + "=" * 70)
    print("MODEL COMPARISON SUMMARY")
    print("=" * 70)
    print(summary_df.to_string(index=False))
    print(f"\nBest model: {best_name}")

    print("\n" + "=" * 70)
    print("PER-LUv2 METRICS — ALL MODELS")
    print("=" * 70)
    print(lu_all_df.to_string(index=False))

    summary_df.to_csv(os.path.join(out_dir, 'summary.csv'), index=False)
    lu_all_df.to_csv(os.path.join(out_dir, 'per_LUv2_metrics_all_models.csv'), index=False)

    fold_rows = []
    for mn, res in all_results.items():
        for fm in res['cv']['fold_metrics']:
            fold_rows.append({'Model': mn, **fm})
    pd.DataFrame(fold_rows).to_csv(os.path.join(out_dir, 'fold_details.csv'), index=False)

    repeat_rows = []
    for mn, res in all_results.items():
        for rs in res['cv']['repeat_summaries']:
            repeat_rows.append({'Model': mn, **rs})
    pd.DataFrame(repeat_rows).to_csv(os.path.join(out_dir, 'repeat_summaries.csv'), index=False)

    for mn, res in all_results.items():
        res['cv']['predictions_df'].to_csv(
            os.path.join(out_dir, f'cv_predictions_{mn}.csv'), index=False)
        res['lu_metrics_df'].to_csv(
            os.path.join(out_dir, f'per_LUv2_metrics_{mn}.csv'), index=False)
        res['cv']['best_params_per_fold'].to_csv(
            os.path.join(out_dir, f'best_params_per_outer_fold_{mn}.csv'), index=False)
        if res['feat_imp']:
            pd.DataFrame(res['feat_imp'], columns=['Feature', 'Importance']).to_csv(
                os.path.join(out_dir, f'feat_importance_{mn}.csv'), index=False)

    models_dir = os.path.join(out_dir, 'models')
    os.makedirs(models_dir, exist_ok=True)
    for mn, res in all_results.items():
        joblib.dump({'model': res['model'], 'scaler': res['scaler'],
                     'encoders': encoders, 'features': feature_cols},
                    os.path.join(models_dir, f'{mn}.pkl'))

    out_excel = os.path.join(out_dir, 'all_results_summary.xlsx')
    with pd.ExcelWriter(out_excel, engine='openpyxl') as writer:
        summary_df.to_excel(writer, sheet_name='Model_summary', index=False)
        lu_all_df.to_excel(writer, sheet_name='PerLUv2_all', index=False)
        pd.DataFrame(fold_rows).to_excel(writer, sheet_name='Fold_details', index=False)
        pd.DataFrame(repeat_rows).to_excel(writer, sheet_name='Repeat_summaries', index=False)
        for mn, res in all_results.items():
            res['cv']['predictions_df'].to_excel(
                writer, sheet_name=f'Preds_{mn[:10]}', index=False)
            res['lu_metrics_df'].to_excel(
                writer, sheet_name=f'LUv2_{mn[:10]}', index=False)
            res['cv']['best_params_per_fold'].to_excel(
                writer, sheet_name=f'Params_{mn[:9]}', index=False)
            if res['feat_imp']:
                pd.DataFrame(res['feat_imp'], columns=['Feature', 'Importance']).to_excel(
                    writer, sheet_name=f'FeatImp_{mn[:8]}', index=False)

    best_res = all_results[best_name]
    with open(os.path.join(out_dir, 'best_model_info.json'), 'w') as f:
        json.dump({
            'best_model': best_name,
            'cv_scheme': f'nested {n_outer_folds}-fold x {n_outer_repeats}-repeat outer, {n_inner_folds}-fold inner',
            'total_outer_fold_runs': n_outer_folds * n_outer_repeats,
            'optuna_trials_per_inner_search': n_trials,
            'overall_r2': float(best_res['cv']['overall']['r2']),
            'overall_rmse': float(best_res['cv']['overall']['rmse']),
            'overall_mae': float(best_res['cv']['overall']['mae']),
            'overall_bias': float(best_res['cv']['overall']['bias']),
            'overall_rpd': float(best_res['cv']['overall']['rpd']),
            'overall_rpiq': float(best_res['cv']['overall']['rpiq']),
            'overall_ccc': float(best_res['cv']['overall']['ccc']),
            'mean_cv_r2': float(best_res['cv']['mean']['r2']),
            'std_cv_r2': float(best_res['cv']['std']['r2_std']),
            'mean_cv_rmse': float(best_res['cv']['mean']['rmse']),
            'std_cv_rmse': float(best_res['cv']['std']['rmse_std']),
            'final_refit_best_params': {k: (v if not isinstance(v, np.generic) else v.item())
                                        for k, v in best_res['best_params'].items()},
            'response_variable': response_variable,
            'soc_upper_limit': soc_upper_limit,
            'n_outer_folds': n_outer_folds,
            'n_outer_repeats': n_outer_repeats,
            'n_inner_folds': n_inner_folds,
            'n_features': len(feature_cols),
            'luv2_classes': sorted(df[luv2_variable].dropna().unique().tolist())
                            if luv2_variable and luv2_variable in df.columns else [],
        }, f, indent=4)

    print(f"\n✅ All outputs saved to : {out_dir}")
    print(f"✅ Excel summary        : {out_excel}")
    return all_results, summary_df, lu_all_df


results, summary, lu_metrics = run_pipeline(
    file_path=INPUT_FILE,
    response_variable=RESPONSE_VARIABLE,
    numeric_preds=NUMERIC_PREDICTORS,
    categorical_preds=CATEGORICAL_PREDICTORS,
    output_base_dir=OUTPUT_BASE_DIR,
    luv2_variable=LUV2_VARIABLE,
    soc_upper_limit=SOC_UPPER_LIMIT,
    n_outer_folds=N_OUTER_FOLDS,
    n_outer_repeats=N_OUTER_REPEATS,
    n_inner_folds=N_INNER_FOLDS,
    n_trials=OPTUNA_TRIALS,
)